### **Projeto END-TO-END de Classificação — Churn da Terra Signal**
#### **Grupo 2 · Hackathon Databricks**

*Objetivo da Terra Signal: uma solução de ponta a ponta para que os usuários de negócio identifiquem os
clientes da carteira **em risco de sair**, e para que o time de vendas ofereça planos de retenção durante
as ligações. Este notebook treina e valida o modelo; a previsão do `inference.csv` vem numa etapa
seguinte.*

**Decisões do modelo** (o racional completo está em `DADOS.md`):

| Decisão | Por quê |
|---|---|
| **Falsos negativos são a prioridade** | Um cliente que sai sem aviso é receita perdida sem chance de reação. Um alarme falso custa só uma ligação de retenção. |
| **Métrica: apenas o `classification_report`** | O foco é o **recall da classe 1 (Saiu)**: de cada 100 clientes que saíram, quantos o modelo encontrou. |
| **Limiares baixos (abaixo de 0,5)** | Baixar o limiar faz o modelo apontar mais clientes como risco, o que reduz os falsos negativos. |
| **`CustomerFeedback` fica de fora** | Parte dos textos foi escrita depois do cancelamento (vazamento do alvo) e o texto não existe para um cliente novo no app. |
| **`MonthlyIncome` fica de fora** | Vem de uma base externa sem garantia de precisão e não muda com o churn. |
| **`TotalCharges` fica de fora** | É multicolinear com `tenure` (Spearman 0,89); sua informação extra já está na `razao_cobranca`. |

---
### **1. Setup Inicial: Importando as Bibliotecas**
*No Databricks, `xgboost` e `shap` podem não vir instalados. A célula abaixo instala os dois.*

In [0]:
%pip install xgboost shap --quiet

In [0]:
# --- Manipulação e Análise de Dados ---
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

# --- Pré-processamento e Estrutura do Modelo ---
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# --- Modelos de Classificação ---
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# --- Métrica de Avaliação ---
from sklearn.metrics import classification_report

# --- Visualização e Interpretabilidade ---
import matplotlib.pyplot as plt
import shap

---
### **2. Coleta de Dados**
*`history.csv` tem a resposta (`Churn`) e serve para treinar e avaliar o modelo. O `inference.csv` só
entra depois que o modelo estiver validado.*

In [0]:
caminho_do_arquivo = "history.csv"
df = pd.read_csv(caminho_do_arquivo)

print(f"{df.shape[0]} linhas x {df.shape[1]} colunas")

---
## **3. Explorar os Dados**

A análise exploratória completa está em `exploratoria.ipynb` e `DADOS.md`. Aqui conferimos só o que afeta
o modelo.

#### **3.1. Tipos de dados e valores vazios**

In [0]:
df.info()

*`tenure` aparece como texto (`str`), não como número, e `TotalCharges` tem valores vazios.*

#### **3.2. Variável alvo**

In [0]:
print(df["Churn"].value_counts(normalize=True).round(3))

*73% ficaram e 27% saíram: as classes são desbalanceadas.*

#### **3.3. Valores das variáveis categóricas**
*Conferimos se alguma categoria aparece escrita de mais de um jeito.*

In [0]:
colunas_texto = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
]

for coluna in colunas_texto:
    print(f"{coluna}: {sorted(df[coluna].unique())}")

*Só `PhoneService` tem problema: aparecem `yes` e `no` em minúsculas, além de `Yes` e `No`.*

#### **3.4. `tenure` com o texto `unknown`**

In [0]:
print(f"tenure = 'unknown': {(df['tenure'] == 'unknown').sum()} linhas")

---
### **4. Limpeza dos Dados**
*Cada problema encontrado na exploração é tratado num passo próprio.*

#### **4.1. `PhoneService`: `yes`/`no` → `Yes`/`No`**
*É só um erro de padronização (confirmado pela organização). Sem tratar, `yes` e `Yes` viram categorias
diferentes no One-Hot Encoding.*

In [0]:
df["PhoneService"] = df["PhoneService"].str.capitalize()

print(df["PhoneService"].value_counts())

#### **4.2. `tenure`: `unknown` → vazio**
*`unknown` é dado perdido na origem. Convertendo a coluna para número, `unknown` vira vazio (`NaN`) e pode
ser imputado.*

In [0]:
df["tenure"] = pd.to_numeric(df["tenure"], errors="coerce")

print(f"tenure vazio: {df['tenure'].isna().sum()} linhas")

#### **4.3. `TotalCharges` vazio com `tenure = 0` → `0`**
*Cliente recém-contratado: ainda não teve fatura fechada, então não pagou nada.*

In [0]:
sem_fatura = df["TotalCharges"].isna() & (df["tenure"] == 0)
df.loc[sem_fatura, "TotalCharges"] = 0

print(f"Clientes sem fatura preenchidos com 0: {sem_fatura.sum()}")

#### **4.4. Imputação de `tenure` vazio → `TotalCharges ÷ MonthlyCharges`**
*`TotalCharges` é basicamente a soma das mensalidades, então dividir pelo valor mensal dá uma estimativa
dos meses de casa (abordagem aprovada pela organização). O resultado é arredondado e limitado a 0–72
meses, porque a empresa tem cerca de 7 anos.*

In [0]:
tenure_vazio = df["tenure"].isna()
tenure_estimado = (df["TotalCharges"] / df["MonthlyCharges"]).round().clip(0, 72)
df.loc[tenure_vazio, "tenure"] = tenure_estimado[tenure_vazio]

print(f"tenure imputado: {tenure_vazio.sum()} linhas")
print(f"tenure ainda vazio: {df['tenure'].isna().sum()} linhas")

#### **4.5. Valores ausentes que seguem para o pipeline**
*Os `TotalCharges` vazios com `tenure > 0` são dado perdido na origem, sem significado de negócio. Eles **não
são imputados aqui**. O `TotalCharges` sai do modelo por multicolinearidade (seção 5.6) e só é usado para
calcular a `razao_cobranca` (seção 5.5): onde ele está vazio, a razão fica vazia e é preenchida pelo
`SimpleImputer` (mediana) dentro do pipeline (seção 8). Assim, a mediana é aprendida só com os dados de treino.*

In [0]:
print("Valores ausentes que irão para o SimpleImputer do pipeline:")
print(df[["tenure", "TotalCharges"]].isna().sum())

---
### **5. Criação de Variáveis**
*Novas variáveis a partir dos padrões encontrados na análise exploratória (`exploratoria.ipynb`). Elas
deixam explícitas combinações que um modelo linear, como a Regressão Logística, não descobre sozinho.
O alvo continua binário (`Yes`/`No`); as variáveis novas do tipo sim/não seguem o mesmo padrão `Yes`/`No`
do restante da base.*

#### **5.1. `n_adicionais`: quantos serviços de proteção e suporte o cliente tem (0 a 4)**
*Na exploratória, entre os clientes com internet, o churn cai de 56% (nenhum adicional) para 6% (os quatro).
Aqui quem não tem internet também fica com 0, e como esse grupo quase não sai (7%), a taxa do 0 fica menor
que a do 1.*

In [0]:
df["n_adicionais"] = (
    (df["OnlineSecurity"] == "Yes").astype(int)
    + (df["OnlineBackup"] == "Yes").astype(int)
    + (df["DeviceProtection"] == "Yes").astype(int)
    + (df["TechSupport"] == "Yes").astype(int)
)

print("Taxa de churn por número de adicionais:")
print((df["Churn"] == "Yes").groupby(df["n_adicionais"]).mean().round(3))

#### **5.2. `inicio_relacionamento`: cliente com até 9 meses de casa**
*9 meses é o primeiro quartil de `tenure`. Nessa fase, o churn é de 49%, quase o dobro da média. A linha
com `tenure` vazio fica vazia aqui também e é preenchida pelo pipeline.*

In [0]:
df["inicio_relacionamento"] = np.where(df["tenure"] <= 9, "Yes", "No")
df.loc[df["tenure"].isna(), "inicio_relacionamento"] = np.nan

print("Taxa de churn por início de relacionamento:")
print((df["Churn"] == "Yes").groupby(df["inicio_relacionamento"]).mean().round(3))

#### **5.3. `mensal_fibra`: contrato mensal e internet de fibra ao mesmo tempo**
*Esse segmento é 30% da base e concentra 62% das saídas.*

In [0]:
df["mensal_fibra"] = np.where(
    (df["Contract"] == "Month-to-month") & (df["InternetService"] == "Fiber optic"), "Yes", "No"
)

print("Taxa de churn por mensal + fibra:")
print((df["Churn"] == "Yes").groupby(df["mensal_fibra"]).mean().round(3))

#### **5.4. `pagamento_automatico`: transferência ou cartão automáticos**
*Quem paga de forma automática sai bem menos (~16%) do que quem paga com cheque eletrônico (45%).*

In [0]:
df["pagamento_automatico"] = np.where(df["PaymentMethod"].str.contains("automatic"), "Yes", "No")

print("Taxa de churn por pagamento automático:")
print((df["Churn"] == "Yes").groupby(df["pagamento_automatico"]).mean().round(3))

#### **5.5. `razao_cobranca`: `TotalCharges ÷ (tenure × MonthlyCharges)`**
*Mostra se a mensalidade mudou ao longo do tempo: perto de 1, o cliente pagou sempre parecido; abaixo de 1,
a mensalidade atual é maior do que a média do passado (aumento ou troca de plano); acima de 1, ela caiu.
Quando `TotalCharges` está vazio ou `tenure = 0`, a razão fica vazia e o pipeline a preenche com a mediana
(≈ 1, ou seja, "sem mudança de plano").*

In [0]:
df["razao_cobranca"] = df["TotalCharges"] / (df["tenure"] * df["MonthlyCharges"])
df["razao_cobranca"] = df["razao_cobranca"].replace([np.inf, -np.inf], np.nan)

print(df["razao_cobranca"].describe().round(3))
print(f"\nrazao_cobranca vazia (vai para o pipeline): {df['razao_cobranca'].isna().sum()} linhas")

#### **5.6. Multicolinearidade: correlação de Spearman entre as variáveis contínuas**
*Variáveis que carregam a mesma informação atrapalham a interpretação: na Regressão Logística os
coeficientes ficam instáveis, e nos modelos de árvore a importância se divide entre elas. Usamos Spearman
porque ele mede relações monotônicas (não só lineares) e não é sensível a outliers. Consideramos
multicolinear um par com |correlação| ≥ 0,8.*

In [0]:
colunas_continuas = ["tenure", "MonthlyCharges", "TotalCharges", "n_adicionais", "razao_cobranca"]

correlacao_spearman = df[colunas_continuas].corr(method="spearman")
correlacao_spearman.round(2)

**Resultado:**
*   **`tenure` × `TotalCharges` = 0,89 → multicolinear.** Era esperado: `TotalCharges` é basicamente
    `tenure × MonthlyCharges`, ou seja, repete a informação das outras duas.
*   As demais correlações vão até 0,66 (moderadas) e fazem sentido de negócio: cada adicional aumenta a
    mensalidade.
*   `razao_cobranca` não se correlaciona com nenhuma outra (≈ 0): traz informação nova.

**Decisão: `TotalCharges` sai do modelo.** Nenhuma informação é perdida, porque
`TotalCharges = razao_cobranca × tenure × MonthlyCharges`: a parte do `TotalCharges` que não é explicada
por `tenure` e `MonthlyCharges` já está na `razao_cobranca`.

---
### **6. Preparação dos Dados**

#### **Separando Variáveis Preditoras (X) e Alvo (y)**
*O alvo vira número: `1` = saiu, `0` = ficou. Ficam fora de `X` o identificador, as duas colunas
descartadas (`CustomerFeedback` e `MonthlyIncome`) e o `TotalCharges` (multicolinear, seção 5.6).*

In [0]:
variavel_alvo = "Churn"

X = df.drop(columns=[variavel_alvo, "customerID", "CustomerFeedback", "MonthlyIncome", "TotalCharges"])
y = (df[variavel_alvo] == "Yes").astype(int)

print(y.value_counts())

#### **Identificando Tipos de Variáveis**

In [0]:
colunas_numericas = [
    "SeniorCitizen", "tenure", "MonthlyCharges",
    "n_adicionais", "razao_cobranca",                                              # novas
]
colunas_categoricas = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
    "inicio_relacionamento", "mensal_fibra", "pagamento_automatico",               # novas
]

---
### **7. Divisão dos Dados**
*O modelo aprende com 80% dos dados e é avaliado nos 20% restantes. `stratify=y` mantém a proporção de
27% de churn nos dois lados.*

In [0]:
x_treino, x_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.2,      # 20% dos dados para teste, 80% para treino
    random_state=42,    # Garante reprodutibilidade na divisão
    stratify=y          # Mantém a proporção das classes
)

---
### **8. Pré-processamento dos Dados**
*   **Numéricas:** preenche vazios com a mediana e padroniza (média 0, desvio padrão 1).
*   **Categóricas:** preenche vazios com a moda e aplica One-Hot Encoding.

*O pré-processador aprende só com o treino (`fit_transform`) e é aplicado no teste (`transform`).*

In [0]:
# --- Pipeline para variáveis numéricas ---
etapas_numericas = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),  # Etapa 1: Preenche valores faltantes com a mediana
    ("scaler", StandardScaler())                   # Etapa 2: Padroniza os dados (média 0, desvio padrão 1)
])

# --- Pipeline para variáveis categóricas ---
etapas_categoricas = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),               # Etapa 1: Preenche com a moda
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))   # Etapa 2: Converte categorias em números
])

# --- Unindo os pipelines ---
preprocessador = ColumnTransformer(transformers=[
    ("num", etapas_numericas, colunas_numericas),
    ("cat", etapas_categoricas, colunas_categoricas)
])

# --- Aplicando as Transformações ---
x_treino_transformado = preprocessador.fit_transform(x_treino)
x_teste_transformado = preprocessador.transform(x_teste)

# Captura os nomes das features para usar na interpretabilidade
nomes_das_features = preprocessador.get_feature_names_out()

---
### **9. Modelagem e Avaliação**

**Como ler o `classification_report`:** o número que mais importa é o **recall da linha `Saiu (1)`**. Os
falsos negativos são os clientes que saíram e o modelo não encontrou: **`(1 − recall) × support`**.

*Nos modelos com `GridSearchCV`, os hiperparâmetros são escolhidos por `scoring="recall"`, coerente com a
prioridade de reduzir os falsos negativos.*

#### **Modelo 0: Baseline - Regressão Logística**
*O ponto de partida mais simples. Os outros modelos precisam superá-lo para justificar a complexidade.*

In [0]:
modelo_lr_baseline = LogisticRegression(random_state=42, max_iter=1000)

print("[INFO] Iniciando o treinamento do Baseline (Regressão Logística)...")
modelo_lr_baseline.fit(x_treino_transformado, y_treino)
print("[INFO] Treinamento do Baseline concluído.")

##### **Avaliação no Conjunto de Treino**

In [0]:
y_pred_treino_modelo_lr_baseline = modelo_lr_baseline.predict(x_treino_transformado)
print("[EVAL] Resultados - Regressão Logística (TREINO):")
print(classification_report(y_treino, y_pred_treino_modelo_lr_baseline, target_names=["Ficou (0)", "Saiu (1)"]))

##### **Avaliação no Conjunto de Teste**

In [0]:
y_pred_teste_modelo_lr_baseline = modelo_lr_baseline.predict(x_teste_transformado)
print("[EVAL] Resultados - Regressão Logística (TESTE):")
print(classification_report(y_teste, y_pred_teste_modelo_lr_baseline, target_names=["Ficou (0)", "Saiu (1)"]))

#### **Modelo 1: Decision Tree Classifier**

In [0]:
modelo_dt = DecisionTreeClassifier(random_state=42)
param_grid_dt = {
    "max_depth": [3, 5, 10],
    "min_samples_split": [2, 10, 20]
}

grid_search_dt = GridSearchCV(estimator=modelo_dt, param_grid=param_grid_dt, cv=5, scoring="recall")
print("[INFO] Iniciando treinamento da Árvore de Decisão...")
grid_search_dt.fit(x_treino_transformado, y_treino)

melhor_modelo_arvore = grid_search_dt.best_estimator_
print(f"[RESULT] Melhores parâmetros: {grid_search_dt.best_params_}")

##### **Avaliação no Conjunto de Treino**

In [0]:
y_pred_treino_melhor_modelo_arvore = melhor_modelo_arvore.predict(x_treino_transformado)
print("[EVAL] Resultados - Árvore de Decisão (TREINO):")
print(classification_report(y_treino, y_pred_treino_melhor_modelo_arvore, target_names=["Ficou (0)", "Saiu (1)"]))

##### **Avaliação no Conjunto de Teste**

In [0]:
y_pred_teste_melhor_modelo_arvore = melhor_modelo_arvore.predict(x_teste_transformado)
print("[EVAL] Resultados - Árvore de Decisão (TESTE):")
print(classification_report(y_teste, y_pred_teste_melhor_modelo_arvore, target_names=["Ficou (0)", "Saiu (1)"]))

#### **Modelo 2: Random Forest Classifier**

In [0]:
modelo_rf = RandomForestClassifier(random_state=42)
param_grid_rf = {
    "n_estimators": [200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 10]
}

grid_search_rf = GridSearchCV(estimator=modelo_rf, param_grid=param_grid_rf, cv=5, scoring="recall")
print("[INFO] Iniciando treinamento do Random Forest...")
grid_search_rf.fit(x_treino_transformado, y_treino)

melhor_modelo_rf = grid_search_rf.best_estimator_
print(f"[RESULT] Melhores parâmetros: {grid_search_rf.best_params_}")

##### **Avaliação no Conjunto de Treino**

In [0]:
y_pred_treino_melhor_modelo_rf = melhor_modelo_rf.predict(x_treino_transformado)
print("[EVAL] Resultados - Random Forest (TREINO):")
print(classification_report(y_treino, y_pred_treino_melhor_modelo_rf, target_names=["Ficou (0)", "Saiu (1)"]))

##### **Avaliação no Conjunto de Teste**

In [0]:
y_pred_teste_melhor_modelo_rf = melhor_modelo_rf.predict(x_teste_transformado)
print("[EVAL] Resultados - Random Forest (TESTE):")
print(classification_report(y_teste, y_pred_teste_melhor_modelo_rf, target_names=["Ficou (0)", "Saiu (1)"]))

#### **Modelo 3: XGBoost Classifier**

In [0]:
modelo_xgb = XGBClassifier(random_state=42)
param_grid_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1]
}

grid_search_xgb = GridSearchCV(estimator=modelo_xgb, param_grid=param_grid_xgb, cv=5, scoring="recall")
print("[INFO] Iniciando treinamento do XGBoost...")
grid_search_xgb.fit(x_treino_transformado, y_treino)

melhor_modelo_xgb = grid_search_xgb.best_estimator_
print(f"[RESULT] Melhores parâmetros: {grid_search_xgb.best_params_}")

##### **Avaliação no Conjunto de Treino**

In [0]:
y_pred_treino_melhor_modelo_xgb = melhor_modelo_xgb.predict(x_treino_transformado)
print("[EVAL] Resultados - XGBoost (TREINO):")
print(classification_report(y_treino, y_pred_treino_melhor_modelo_xgb, target_names=["Ficou (0)", "Saiu (1)"]))

##### **Avaliação no Conjunto de Teste**

In [0]:
y_pred_teste_melhor_modelo_xgb = melhor_modelo_xgb.predict(x_teste_transformado)
print("[EVAL] Resultados - XGBoost (TESTE):")
print(classification_report(y_teste, y_pred_teste_melhor_modelo_xgb, target_names=["Ficou (0)", "Saiu (1)"]))

---
### **10. Análise de Limiar (Threshold)**

Por padrão, o modelo diz "vai sair" quando a probabilidade é **≥ 0,5**. Como os falsos negativos são a
prioridade, testamos **limiares mais baixos**: o modelo passa a apontar como risco clientes com
probabilidade menor, o que encontra mais clientes que saem (**recall sobe, falsos negativos caem**) ao custo
de mais alarmes falsos (**precision cai**).

*É como deixar o alarme de incêndio mais sensível: ele toca mais vezes à toa, mas quase não deixa passar
um incêndio de verdade.*

In [0]:
limiares = [0.50, 0.40, 0.35, 0.30, 0.25, 0.20, 0.15]

#### **Modelo 0: Regressão Logística (Baseline)**

In [0]:
print("[INFO] Análise de limiar - Modelo 0: Regressão Logística (Baseline)")

y_probas_teste_lr_baseline = modelo_lr_baseline.predict_proba(x_teste_transformado)[:, 1]

for limiar in limiares:
    y_pred_limiar = (y_probas_teste_lr_baseline >= limiar).astype(int)
    print(f"\n[EVAL] Modelo 0: Regressão Logística (Baseline) (TESTE, limiar {limiar:.2f}):")
    print(classification_report(y_teste, y_pred_limiar, target_names=["Ficou (0)", "Saiu (1)"]))

#### **Modelo 1: Decision Tree Classifier**

In [0]:
print("[INFO] Análise de limiar - Modelo 1: Decision Tree Classifier")

y_probas_teste_arvore = melhor_modelo_arvore.predict_proba(x_teste_transformado)[:, 1]

for limiar in limiares:
    y_pred_limiar = (y_probas_teste_arvore >= limiar).astype(int)
    print(f"\n[EVAL] Modelo 1: Decision Tree Classifier (TESTE, limiar {limiar:.2f}):")
    print(classification_report(y_teste, y_pred_limiar, target_names=["Ficou (0)", "Saiu (1)"]))

#### **Modelo 2: Random Forest Classifier**

In [0]:
print("[INFO] Análise de limiar - Modelo 2: Random Forest Classifier")

y_probas_teste_rf = melhor_modelo_rf.predict_proba(x_teste_transformado)[:, 1]

for limiar in limiares:
    y_pred_limiar = (y_probas_teste_rf >= limiar).astype(int)
    print(f"\n[EVAL] Modelo 2: Random Forest Classifier (TESTE, limiar {limiar:.2f}):")
    print(classification_report(y_teste, y_pred_limiar, target_names=["Ficou (0)", "Saiu (1)"]))

#### **Modelo 3: XGBoost Classifier**

In [0]:
print("[INFO] Análise de limiar - Modelo 3: XGBoost Classifier")

y_probas_teste_xgb = melhor_modelo_xgb.predict_proba(x_teste_transformado)[:, 1]

for limiar in limiares:
    y_pred_limiar = (y_probas_teste_xgb >= limiar).astype(int)
    print(f"\n[EVAL] Modelo 3: XGBoost Classifier (TESTE, limiar {limiar:.2f}):")
    print(classification_report(y_teste, y_pred_limiar, target_names=["Ficou (0)", "Saiu (1)"]))

#### **Resumo (conjunto de teste: 301 clientes que saíram)**

| Modelo | Recall `Saiu` (0,50) | Recall `Saiu` (0,20) | Precision `Saiu` (0,20) | Falsos negativos (0,20) |
|---|---|---|---|---|
| Regressão Logística | 0,52 | 0,86 | 0,47 | ~42 |
| Árvore de Decisão | 0,62 | 0,85 | 0,50 | ~45 |
| **Random Forest** | 0,51 | **0,87** | 0,48 | **~39** |
| XGBoost | 0,53 | 0,83 | 0,49 | ~51 |

**Efeito das variáveis novas (seção 5) e da retirada do `TotalCharges` (seção 5.6):** com limiar 0,20, a
Árvore de Decisão foi a que mais ganhou (recall de 0,70 → 0,85) e ficou mais rasa (`max_depth` de 10 → 5),
ou seja, decora menos o treino. Nos outros modelos, a mudança foi de ±2 pontos.

**Escolha: Random Forest com limiar 0,20.**
*   No limiar padrão, os quatro modelos deixam passar quase metade dos clientes que saem.
*   Com limiar 0,20, o Random Forest encontra **quase 9 de cada 10** clientes que saem: os falsos negativos
    caem de **~147 para ~39**.
*   O custo: cerca de metade dos clientes apontados não iria sair (precision 0,48). Para uma ligação de
    retenção, esse é um custo aceitável perto de perder o cliente.
*   A diferença para a Regressão Logística (0,86) é de ~3 clientes em 301, dentro do ruído de uma única
    divisão treino/teste. A Regressão Logística é uma alternativa igualmente válida e mais simples de explicar.
*   Abaixo de 0,20, o ganho de recall é pequeno e os alarmes falsos continuam crescendo.

*Os números podem variar um pouco entre execuções e versões de biblioteca. Se mudarem, revise o resumo e
o `limiar_escolhido` abaixo.*

---
### **11. O modelo vale a pena? Comparativo com estratégias sem modelo**

Um modelo só se justifica se trouxer **mais resultado do que a empresa conseguiria sem ele**. Comparamos o
Random Forest (limiar 0,20) com quatro alternativas, no mesmo conjunto de teste (1.127 clientes, 301 que
saíram):

| Estratégia | O que representa |
|---|---|
| Não fazer nada | O cenário de hoje: nenhuma ação de retenção |
| Contatar todos | Ação em massa, sem critério |
| Aleatório | Sorteia o **mesmo número de clientes** que o modelo aponta: mostra se o modelo escolhe melhor que o acaso |
| Regra simples | Aborda todo cliente com contrato mensal: o "bom senso" sem modelo |
| Modelo | Aborda os clientes que o Random Forest aponta como risco |

#### **11.1. Premissas financeiras**

A ideia é simples: a Terra Signal quer ligar para os clientes que estão pensando em sair e oferecer algo
para que fiquem. Ligar custa dinheiro e não funciona sempre. Para saber se vale a pena, é preciso responder
três perguntas, e cada premissa é a resposta de uma delas:

| Pergunta | Premissa | Por quê |
|---|---|---|
| **Quanto a empresa ganha se o cliente ficar?** | **12 mensalidades** | Quando o cliente cancela, a empresa para de receber a mensalidade dele; quando fica, continua recebendo. Não dá para saber por quanto tempo ele vai continuar, então supomos **um ano**. É conservador (há clientes de até 72 meses) e combina com a ação de retenção: quem migra para o **plano anual** fica pelo menos 12 meses por contrato. |
| **Quanto custa tentar segurar um cliente?** | **R$ 50 por cliente abordado** | O tempo do atendente na ligação mais o benefício oferecido (um desconto, por exemplo), abaixo de uma mensalidade média (~R$ 65). **É pago para todo cliente abordado, inclusive os que iam ficar de qualquer jeito** (os alarmes falsos). |
| **De cada 10 clientes que iam sair e foram abordados, quantos ficam?** | **30% (3 de cada 10)** | Um cenário otimista, explicado abaixo. |

**De onde vêm esses valores.** A organização não forneceu nenhum deles: disse que não tem um custo padrão e
pediu que a equipe **definisse as premissas** e mostrasse que o retorno supera o custo (*R22*, *R23*). As 12
mensalidades já tinham sido adotadas na análise exploratória (Ato 4). Os R$ 50 e os 30% são **suposições da
equipe**: a base não traz o custo nem o resultado de nenhuma ação de retenção.

**Sobre a taxa de sucesso.** Existem dois tipos de abordagem:
*   **Reativa:** o próprio cliente liga dizendo "quero cancelar" e o atendente tenta convencê-lo. Casos
    publicados por empresas de atendimento mostram taxas que vão de menos de 10% a cerca de 70%, conforme a
    oferta e a forma de contar o sucesso ([Qualfon](https://www.qualfon.com/case-studies/telecom-customer-retention-160-percent-improvement),
    [Foundever](https://foundever.com/case-studies/u-s-telecoms-provider-surpasses-its-customer-retention-targets-in-just-five-months/)).
*   **Proativa (o nosso caso):** a empresa liga **antes**, para quem o modelo apontou, mesmo que o cliente
    nunca tenha falado em cancelar. Tende a funcionar menos. Num experimento com 65 mil clientes de uma
    operadora sul-americana, oferecer um plano novo **aumentou** o churn de 6% para 10%, porque a ligação
    "acordou" clientes que não pensavam em sair ([Iyengar e Schleicher, Columbia Business School](https://business.columbia.edu/faculty/research/perils-proactive-churn-prevention-using-plan-recommendations-evidence-field)).
    A conclusão dos autores é que a ação só funciona quando é bem direcionada.

Por isso os 30% são um cenário **otimista**. O passo a passo da seção 11.2 calcula a partir de que taxa o
modelo deixa de se pagar.

**Limitações:**
*   As 12 mensalidades são **receita, não lucro**: não descontam o custo de atender o cliente.
*   Elas supõem que o cliente fica exatamente mais um ano; se ficar mais, o ganho real é maior.
*   **Próximo passo recomendado: um piloto com grupo de controle.** No primeiro mês, aborda-se metade dos
    clientes apontados pelo modelo e a outra metade fica sem contato. Comparando os dois grupos, a Terra
    Signal mede a **sua própria** taxa de sucesso e troca a suposição por um número real.

*As premissas ficam nas variáveis abaixo, para serem ajustadas.*

In [0]:
meses_de_valor = 12          # valor de um cliente retido = 12 mensalidades
custo_por_contato = 50.0     # R$ por cliente abordado (ligação + benefício)
taxa_de_sucesso = 0.30       # fração dos clientes que iam sair e ficam após a abordagem

#### **11.2. Exemplo passo a passo com os nossos dados**
*A conta completa para o modelo, no conjunto de teste: começa por um cliente e termina no retorno de toda
a estratégia. No fim, os pontos de equilíbrio mostram até onde as premissas podem piorar sem o modelo dar
prejuízo.*

**As fórmulas:**

```
Valor de um cliente retido = mensalidade × 12
Receita preservada         = (soma do valor dos clientes que iam sair e foram abordados) × taxa de sucesso
Custo das ações            = clientes abordados × custo por contato
Retorno líquido            = receita preservada − custo das ações
ROI                        = retorno líquido ÷ custo das ações
```

*   O **retorno líquido** diz **quanto sobra em reais**.
*   O **ROI** (*Return on Investment*, retorno sobre o investimento) diz **quanto sobra para cada R$ 1
    gasto**. Um ROI de 100% quer dizer que cada R$ 1 gasto volta como R$ 2: o R$ 1 investido mais R$ 1 de
    ganho. ROI negativo quer dizer prejuízo.
*   Os dois se complementam: o retorno líquido mostra o **tamanho** do ganho, e o ROI mostra a **eficiência**
    do dinheiro gasto.

In [0]:
y_pred_modelo = (y_probas_teste_rf >= 0.20).astype(int)
saiu = y_teste.to_numpy() == 1
mensalidade_teste = x_teste["MonthlyCharges"].to_numpy()

# --- Passo 1: quanto vale segurar um cliente ---
encontrado = (y_pred_modelo == 1) & saiu
indice_exemplo = x_teste.index[encontrado][0]
mensalidade_exemplo = df.loc[indice_exemplo, "MonthlyCharges"]
valor_exemplo = meses_de_valor * mensalidade_exemplo
print("PASSO 1 - Quanto vale segurar um cliente")
print(f"  O cliente {df.loc[indice_exemplo, 'customerID']} paga R$ {mensalidade_exemplo:.2f} por mês, ".replace(".", ",")
      + "foi apontado pelo modelo e de fato saiu.")
print(f"  Se ele ficasse mais {meses_de_valor} meses, a empresa receberia "
      f"{meses_de_valor} x R$ {mensalidade_exemplo:.2f} = R$ {valor_exemplo:.2f}".replace(".", ",") + ".\n")

# --- Passo 2: para quem o modelo manda ligar ---
n_contatos = y_pred_modelo.sum()
n_iam_sair = encontrado.sum()
print("PASSO 2 - Para quem o modelo manda ligar")
print(f"  {n_contatos} de {len(y_teste)} clientes. Desses, {n_iam_sair} iam sair de verdade "
      f"e {n_contatos - n_iam_sair} iam ficar de qualquer jeito (alarmes falsos).\n")

# --- Passo 3: quantos ficam depois da ligação ---
print("PASSO 3 - Quantos ficam depois da ligação")
print(f"  {n_iam_sair} clientes que iam sair x {taxa_de_sucesso:.0%} de sucesso = "
      f"{n_iam_sair * taxa_de_sucesso:.0f} clientes mantidos, em média.\n")

# --- Passo 4: dinheiro que fica na empresa ---
valor_dos_que_iam_sair = (mensalidade_teste[encontrado] * meses_de_valor).sum()
receita_preservada = valor_dos_que_iam_sair * taxa_de_sucesso
print("PASSO 4 - Dinheiro que fica na empresa")
print(f"  {meses_de_valor} mensalidades dos {n_iam_sair} que iam sair = R$ {valor_dos_que_iam_sair:,.0f}".replace(",", "."))
print(f"  x {taxa_de_sucesso:.0%} de sucesso = R$ {receita_preservada:,.0f}\n".replace(",", "."))

# --- Passo 5: dinheiro gasto nas ligações ---
custo = n_contatos * custo_por_contato
print("PASSO 5 - Dinheiro gasto nas ligações")
print(f"  {n_contatos} clientes abordados x R$ {custo_por_contato:.0f} = R$ {custo:,.0f}\n".replace(",", "."))

# --- Passo 6: o que sobra ---
retorno_liquido = receita_preservada - custo
print("PASSO 6 - O que sobra (retorno líquido = dinheiro mantido - dinheiro gasto)")
print(f"  R$ {receita_preservada:,.0f} - R$ {custo:,.0f} = R$ {retorno_liquido:,.0f}\n".replace(",", "."))

# --- Passo 7: ROI ---
roi = retorno_liquido / custo
print("PASSO 7 - ROI (retorno líquido / dinheiro gasto)")
print(f"  R$ {retorno_liquido:,.0f} / R$ {custo:,.0f} = ".replace(",", ".") + f"{roi:.1%}".replace(".", ","))
print(f"  Cada R$ 1 gasto nas ligações volta como R$ {1 + roi:.2f}: ".replace(".", ",")
      + f"o R$ 1 investido mais R$ {roi:.2f} de ganho".replace(".", ",") + ".\n")

# --- Pontos de equilíbrio: até onde as premissas podem piorar ---
taxa_minima = custo / valor_dos_que_iam_sair
print("PONTOS DE EQUILÍBRIO - a partir de onde o modelo deixa de se pagar")
print(f"  Com custo de R$ {custo_por_contato:.0f}: a taxa de sucesso precisa ser de pelo menos "
      + f"{taxa_minima:.1%}".replace(".", ",")
      + f" (cerca de 1 em cada {1 / taxa_minima:.0f} clientes).")
print(f"  Com {taxa_de_sucesso:.0%} de sucesso: o custo por cliente abordado pode chegar a "
      f"R$ {receita_preservada / n_contatos:.0f}.")

#### **11.3. Modelo × sorteio aleatório**
*O sorteio escolhe ao acaso o mesmo número de clientes que o modelo aponta. A comparação usa o mesmo
`classification_report` das seções anteriores.*

In [0]:
n_contatos = y_pred_modelo.sum()

gerador = np.random.default_rng(42)
y_pred_aleatorio = np.zeros(len(y_teste), dtype=int)
y_pred_aleatorio[gerador.choice(len(y_teste), size=n_contatos, replace=False)] = 1

print(f"Clientes apontados como risco: {n_contatos} de {len(y_teste)}\n")
print("[EVAL] Sorteio aleatório (TESTE):")
print(classification_report(y_teste, y_pred_aleatorio, target_names=["Ficou (0)", "Saiu (1)"]))
print("[EVAL] Random Forest, limiar 0.20 (TESTE):")
print(classification_report(y_teste, y_pred_modelo, target_names=["Ficou (0)", "Saiu (1)"]))

#### **11.4. Retorno financeiro de cada estratégia**
*Para cada estratégia:*
*   **Receita preservada** = soma de 12 mensalidades dos clientes que iam sair e foram abordados × taxa de sucesso
*   **Custo das ações** = clientes abordados × custo por contato
*   **Retorno líquido** = receita preservada − custo das ações
*   **ROI** = retorno líquido ÷ custo das ações (em "Não fazer nada" não há gasto, então não há ROI)



In [0]:
linhas = []
for nome, contatar in estrategias.items():
    encontrados = contatar & saiu
    receita_preservada = (mensalidade_teste[encontrados] * meses_de_valor).sum() * taxa_de_sucesso
    custo = contatar.sum() * custo_por_contato
    retorno_liquido = receita_preservada - custo
    roi = (100 * retorno_liquido / custo) if custo > 0 else None

    linhas.append({
        "Estratégia": nome,
        "Clientes abordados": int(contatar.sum()),
        "Iam sair e foram abordados": int(encontrados.sum()),
        "Receita preservada (R$)": round(receita_preservada),
        "Custo das ações (R$)": round(custo),
        "Retorno líquido (R$)": round(retorno_liquido),
        "ROI (%)": round(roi, 1) if roi is not None else None,
    })

comparativo = pd.DataFrame(linhas).set_index("Estratégia")

# Formatação visual para exibição na tabela do notebook
comparativo_exibicao = comparativo.copy()
for col in ["Receita preservada (R$)", "Custo das ações (R$)", "Retorno líquido (R$)"]:
    comparativo_exibicao[col] = comparativo_exibicao[col].apply(
        lambda v: f"R$ {v:,.0f}".replace(",", ".")
    )
comparativo_exibicao["ROI (%)"] = comparativo_exibicao["ROI (%)"].apply(
    lambda v: f"{v:.1f}%".replace(".", ",") if pd.notnull(v) else "—"
)

comparativo_exibicao

**Como ler as colunas financeiras da tabela:**

* **Receita preservada (R$):** É o faturamento bruto que a empresa deixa de perder graças à retenção. A conta soma 12 mensalidades individuais apenas dos clientes que **realmente iam cancelar e foram abordados** (verdadeiros positivos) e multiplica pela taxa de sucesso da ligação (30%). Clientes abordados por engano (alarmes falsos) não geram receita incremental aqui, pois já continuariam pagando normalmente.
* **Custo das ações (R$):** É o gasto total da operação de retenção. Multiplica o custo unitário da abordagem (R$ 50) por **todos os clientes contatados** — cobrando tanto as ligações certas quanto o desperdício com os alarmes falsos.
* **Retorno líquido (R$):** É a diferença entre o que entrou e o que saiu (`Receita preservada - Custo das ações`), mostrando quanto dinheiro efetivamente sobra no caixa.
* **ROI (%):** É a razão entre o retorno líquido e o custo (`Retorno líquido / Custo das ações`), indicando a eficiência de cada real investido na operação. (em "Não fazer nada" não há gasto, então não há ROI)

**O que essas duas colunas revelam na comparação com a regra simples:**
Embora a regra de contrato mensal encontre 5 clientes a mais que iam sair (266 contra 261), o modelo preserva **R$ 1.410 a mais em receita** (R$ 72.411 contra R$ 71.001) porque prioriza clientes de mensalidade mais alta. Ao mesmo tempo, reduz o **custo das ações em R$ 3.750** ao fazer 75 ligações a menos — explicando a vantagem total de **R$ 5.160** no retorno líquido.

#### **11.5. Retorno líquido do modelo em cada limiar**
*Mesmas premissas, variando só o limiar. Mostra se o limiar 0,20, escolhido para reduzir os falsos
negativos, também faz sentido financeiramente.*

In [0]:
linhas = []
for limiar in limiares:
    contatar = y_probas_teste_rf >= limiar
    encontrados = contatar & saiu
    receita_preservada = (mensalidade_teste[encontrados] * meses_de_valor).sum() * taxa_de_sucesso
    custo = contatar.sum() * custo_por_contato
    retorno_liquido = receita_preservada - custo
    roi = (100 * retorno_liquido / custo) if custo > 0 else None

    linhas.append({
        "Limiar": limiar,
        "Clientes abordados": int(contatar.sum()),
        "Iam sair e foram abordados": int(encontrados.sum()),
        "Receita preservada (R$)": round(receita_preservada),
        "Custo das ações (R$)": round(custo),
        "Retorno líquido (R$)": round(retorno_liquido),
        "ROI (%)": round(roi, 1) if roi is not None else None,
    })

df_limiares = pd.DataFrame(linhas).set_index("Limiar")

# Formatação visual no padrão brasileiro para exibição no notebook
df_limiares_exibicao = df_limiares.copy()
df_limiares_exibicao.index = [f"{v:.2f}".replace(".", ",") for v in df_limiares_exibicao.index]
df_limiares_exibicao.index.name = "Limiar"

for col in ["Receita preservada (R$)", "Custo das ações (R$)", "Retorno líquido (R$)"]:
    df_limiares_exibicao[col] = df_limiares_exibicao[col].apply(
        lambda v: f"R$ {v:,.0f}".replace(",", ".")
    )

df_limiares_exibicao["ROI (%)"] = df_limiares_exibicao["ROI (%)"].apply(
    lambda v: f"{v:.1f}%".replace(".", ",") if pd.notnull(v) else "—"
)

df_limiares_exibicao

**Como interpretar a evolução dos limiares:**

* **Descendo de 0,50 para 0,20:** O **Custo das ações** sobe porque abordamos mais clientes de risco moderado (incluindo mais alarmes falsos), o que reduz o **ROI percentual** por real investido. Porém, a **Receita preservada** cresce num ritmo ainda maior do que o custo — fazendo o **Retorno líquido** atingir o pico máximo exatamente no **limiar 0,20**.
* **Abaixo de 0,20 (ex.: 0,15):** O custo de abordar clientes de baixo risco passa a empatar ou superar a receita extra recuperada, estagnando o retorno líquido e derrubando o ROI sem ganho real para o caixa.

#### **11.6. Conclusão**

| Estratégia | Clientes abordados | Iam sair e foram abordados | Retorno líquido | ROI |
|---|---|---|---|---|
| Não fazer nada | 0 | 0 | R$ 0 | — |
| Aleatório | 547 | 141 | R$ 11.275 | 41% |
| Contatar todos | 1.127 | 301 | R$ 25.011 | 44% |
| Regra simples: contrato mensal | 622 | 266 | R$ 39.901 | 128% |
| **Modelo (limiar 0,20)** | **547** | **261** | **R$ 45.061** | **165%** |

*   **Melhor que o acaso:** com o mesmo número de contatos, o modelo encontra **261** clientes que iam sair,
    contra **141** do sorteio. A precision da classe `Saiu` sobe de 0,26 (a taxa de churn da base) para 0,48,
    e o retorno líquido é **4 vezes maior**.
*   **Melhor que contatar todos:** o modelo aborda metade dos clientes e mesmo assim encontra 87% dos que iam
    sair. Economiza R$ 29.000 em ações desnecessárias e rende **R$ 20.050 a mais**.
*   **Melhor que a regra simples, mas por pouco:** a regra "contrato mensal" já é boa (R$ 39.901). O modelo
    rende **R$ 5.160 a mais (+13%)** com **75 contatos a menos**. O ganho existe, mas é moderado: a maior parte
    do sinal está no tipo de contrato.
*   **O dinheiro mais bem gasto:** o modelo tem o maior ROI entre as estratégias. Cada R$ 1 gasto nas
    ligações volta como **R$ 2,65**, contra R$ 2,28 da regra simples e R$ 1,44 de contatar todos.
*   **O limiar 0,20 também é o melhor financeiramente:** entre os limiares testados, é o que dá o maior retorno
    líquido (o 0,15 empata). A escolha feita para reduzir os falsos negativos coincide com a que dá mais dinheiro.

**Por que o ROI é maior no limiar 0,50, se o retorno líquido é maior no 0,20?** Com limiar alto, o modelo só
aponta os clientes de risco mais evidente: gasta pouco e acerta muito, por isso o ROI é alto (272%). Ao baixar
o limiar, entram clientes de risco menos evidente, e cada um deles rende menos por real gasto, então o ROI cai.
Mas **cada contato extra ainda rende mais do que custa**, por isso o dinheiro que sobra no total continua
subindo até 0,20. Na prática:
*   Se a meta é **ganhar o máximo de dinheiro** (e reduzir os falsos negativos), o limiar é **0,20**.
*   Se a equipe de retenção tiver **orçamento ou capacidade limitada**, um limiar mais alto aborda menos
    clientes com o dinheiro mais bem aproveitado.

**Cuidados:**
*   Os valores dependem das premissas (11.1). Com um custo por contato maior ou uma taxa de sucesso menor, a
    vantagem de abordar poucos clientes certos (o modelo) sobre abordar muitos (contatar todos) aumenta.
*   Os valores são para os 1.127 clientes do conjunto de teste, numa única divisão treino/teste. O sorteio
    aleatório também é uma única sorteada (`random_state=42`); em média, ele encontra 27% de clientes que iam sair.

---
### **12. Modelo Final**

#### **12.1. Treinamento com todo o histórico**
*O modelo escolhido é treinado de novo com os 5.634 clientes do `history`, dentro de um único `Pipeline`
(pré-processamento + modelo). Assim, o app só precisa passar os dados limpos e recebe a probabilidade.*

In [0]:
limiar_escolhido = 0.20

modelo_final = Pipeline(steps=[
    ("preprocessador", clone(preprocessador)),
    ("modelo", clone(melhor_modelo_rf))
])

print("[INFO] Treinando o modelo final com todo o histórico...")
modelo_final.fit(X, y)
print("[INFO] Modelo final treinado.")

#### **12.2. Salvando o modelo**
*O arquivo guarda o pipeline treinado, o limiar e a lista de colunas que ele espera receber. Ele será usado
na previsão do `inference.csv` e no app. No Databricks, a pasta do repositório não aceita escrita, então o
arquivo vai para a pasta do usuário; fora do Databricks, vai para a pasta atual.*

In [0]:
artefato = {
    "modelo": modelo_final,
    "limiar": limiar_escolhido,
    "colunas": list(X.columns),
}

try:
    from databricks.sdk import WorkspaceClient
    pasta_de_saida = Path(f"/Workspace/Users/{WorkspaceClient().current_user.me().user_name}")
except Exception:
    pasta_de_saida = Path(".")

caminho_modelo = pasta_de_saida / "modelo_churn.joblib"
joblib.dump(artefato, caminho_modelo)
print(f"[INFO] Modelo salvo em: {caminho_modelo}")

---
### **13. Interpretabilidade**

#### **13.1. Feature Importance**
*Quanto cada variável pesa nas decisões do Random Forest.*

In [0]:
importancias = modelo_final.named_steps["modelo"].feature_importances_
nomes_finais = modelo_final.named_steps["preprocessador"].get_feature_names_out()

df_importancias = pd.DataFrame({
    "Variaveis": nomes_finais,
    "Importancia": importancias
}).sort_values(by="Importancia", ascending=False)

df_importancias.head(15)

#### **13.2. Shapley Value**
*Mostra, para cada variável, se ela empurra a previsão para "sai" (direita) ou "fica" (esquerda). Cada ponto
é um cliente; a cor indica se o valor da variável é alto (vermelho) ou baixo (azul). Usamos uma amostra de
500 clientes do teste para a conta ser rápida.*

In [0]:
amostra_teste = x_teste_transformado[:500]

explainer_rf = shap.TreeExplainer(melhor_modelo_rf)
shap_values_rf = explainer_rf.shap_values(amostra_teste)
shap_values_classe_rf = shap_values_rf[:, :, 1]   # Classe 1 = Saiu

shap.summary_plot(
    shap_values_classe_rf,
    amostra_teste,
    feature_names=nomes_das_features,
    plot_type="dot",
    show=False
)
plt.title("SHAP Beeswarm Plot - Random Forest (Classe Saiu)")
plt.show()